# SyncManager v1.2 Routing

## What this notebook proves
- `freshness("stale_ok")` routes reads to local when policy permits.
- `freshness("realtime")` routes reads to remote.
- `freshness("auto")` routes by lag/circuit policy.

## Deterministic expectation in this demo
- local has 2 rows (`local-a`, `local-b`)
- remote has 1 row (`remote-a`)
- stale_ok -> 2 rows
- realtime -> 1 row
- auto (healthy lag) -> 2 rows


In [ ]:
from surrealengine import (
    Document,
    StringField,
    create_connection,
    create_sync_manager,
    SyncConfig,
    SyncPolicy,
)

class User(Document):
    name = StringField()

    class Meta:
        collection = "users_sync_routing"


In [ ]:
import os

SURREAL_URL = os.getenv("SURREAL_URL", "ws://localhost:8080/rpc")
SURREAL_NS = os.getenv("SURREAL_NS", "demo")
SURREAL_DB = os.getenv("SURREAL_DB", "sync_demo")
SURREAL_USER = os.getenv("SURREAL_USER", "root")
SURREAL_PASS = os.getenv("SURREAL_PASS", "secret")

# Remote source of truth
remote = create_connection(
    url=SURREAL_URL,
    namespace=SURREAL_NS,
    database=SURREAL_DB,
    username=SURREAL_USER,
    password=SURREAL_PASS,
    make_default=True,
)
await remote.connect()

# Local low-latency read model
local = create_connection(
    url="mem://tmp/sync_local?snapshot=60s&sync=5s",
    namespace=SURREAL_NS,
    database=SURREAL_DB,
)
await local.connect()

sync = create_sync_manager(
    remote=remote,
    local=local,
    config=SyncConfig(auto_max_lag_ms=750),
    make_default=True,
)
sync.register_model(User, policy=SyncPolicy.mirrored)

# Ensure collection exists on both sides and seed deterministic data
await local.client.query("DEFINE TABLE users_sync_routing SCHEMALESS;")
await remote.client.query("DEFINE TABLE users_sync_routing SCHEMALESS;")
await local.client.query("DELETE users_sync_routing;")
await remote.client.query("DELETE users_sync_routing;")
await User(id="users_sync_routing:local_a", name="local-a").save(connection=local)
await User(id="users_sync_routing:local_b", name="local-b").save(connection=local)
await User(id="users_sync_routing:remote_a", name="remote-a").save(connection=remote)

print(sync.status_sync())


In [ ]:
print("Connected remote:", remote.client is not None)
print("Connected local:", local.client is not None)
print("Initial SyncManager status:", sync.status_sync())


In [ ]:
# Route local-first
users_local = await User.objects.freshness("stale_ok").all()
print("stale_ok count:", len(users_local), [u.name for u in users_local])

# Route remote-first
users_remote = await User.objects.freshness("realtime").all()
print("realtime count:", len(users_remote), [u.name for u in users_remote])

# Auto mode (lag/circuit aware) with healthy lag should route local
sync.set_lag_ms(100)
users_auto = await User.objects.freshness("auto").all()
print("auto count (healthy lag):", len(users_auto), [u.name for u in users_auto])


In [ ]:
checks = {
    "stale_ok_count_is_2": isinstance(users_local, list) and len(users_local) == 2,
    "realtime_count_is_1": isinstance(users_remote, list) and len(users_remote) == 1,
    "auto_count_is_2": isinstance(users_auto, list) and len(users_auto) == 2,
}
for name, ok in checks.items():
    print(f"{name}: {'PASS' if ok else 'FAIL'}")

if not all(checks.values()):
    raise AssertionError("Routing deterministic checks failed")

print("ALL ROUTING CHECKS PASSED")


In [ ]:
# Cleanup
await remote.disconnect()
await local.disconnect()
